# 2️⃣ From Zero to a Working DBT
### 🎯 Interactive Workshop Notebook

**Welcome!** In this notebook, you’ll turn your **tool setup** into a **working DBT project**.

## What you’ll do
- ✅ Check that you opened the notebook in the right repo
- ✅ Create a project virtual environment
- ✅ Install the workshop packages
- ✅ Build a DBT project step by step
- ✅ Connect DBT to DuckDB
- ✅ Load data and run first model
- ✅ Query the results you created

This notebook should be opened from inside the cloned workshop repository.

## How to use this notebook
1. Read the short explanation
2. Run the cell below it ▶️
3. Check the result against the checkpoint ✅
4. If something breaks, use the troubleshooting section at the end

> **Good news:** this notebook avoids tricky OS-specific commands where possible.
> It uses Python paths directly, so it works more reliably on **Windows, Linux, and Chromebook Linux**.


## 🔍 Step 1 — Check your setup and find the repo

This cell does three important things:
- shows your Python + operating system
- finds the **repo root** automatically
- checks that the `dbt_project/` folder exists


In [6]:
from pathlib import Path
import os, platform, re, subprocess, sys, textwrap

#--------------------------------------------#
def find_repo_root(start: Path) -> Path:
  candidates = [start] + list(start.parents)

  for candidate in candidates:
      if (
          (candidate / "README.md").exists()
          and (candidate / "requirements.txt").exists()
          and (candidate / "02_environment").exists()
      ):
          return candidate

  raise FileNotFoundError(
      "Could not find the workshop repo root. "
      "Open this notebook from inside the cloned dbt-workshop repo."
  )

#--------------------------------------------#
#  Determine OS, folder names and file names 
#--------------------------------------------#
OS_NAME = platform.system()

repo_root = find_repo_root(Path.cwd())

venv_dir = repo_root / ".venv"

venv_python = venv_dir / (
  "Scripts/python.exe" if OS_NAME == "Windows" else "bin/python"
)

venv_pip = venv_dir / (
  "Scripts/pip.exe" if OS_NAME == "Windows" else "bin/pip"
)

venv_dbt = venv_dir / (
  "Scripts/dbt.exe" if OS_NAME == "Windows" else "bin/dbt"
)

requirements_file = repo_root / (
  "requirements_locked.txt"
  if (repo_root / "requirements_locked.txt").exists()
  else "requirements.txt"
)

project_name = "p00_setup"
dbt_project_dir = repo_root / "dbt_projects" / project_name

#-------------------------------------------------#
# Print out OS, Python Version, Repo Root Folder
# Project Folder and Installation Requirement File
#-------------------------------------------------#
print(f"Python: {sys.version.split()[0]}")
print(f"Operating System: {OS_NAME}")

print(f"Repo root: {repo_root}")

if dbt_project_dir.exists():
    print("ℹ️ A dbt_project folder already exists.")
else:
    print("ℹ️ No dbt_project folder yet — it will be created in this notebook.")

print(f"Requirements file: {requirements_file.name}")


Python: 3.11.2
Operating System: Linux
Repo root: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop
ℹ️ A dbt_project folder already exists.
Requirements file: requirements_locked.txt


### ✅ Checkpoint
You should see:

- a **Python version**
- your **operating system**
- the path to your **repo root**
- the active **requirements file** that contains a list of versioned applications that will be installed


## Step 2 — Helper function

This helper runs terminal commands and shows the output inside the notebook.

---

In [7]:
def run_command(cmd, cwd=None, check=True):
    print("Running:", " ".join(str(x) for x in cmd))
    completed = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True
    )

    if completed.stdout.strip():
        print(completed.stdout)

    if completed.stderr.strip():
        print(completed.stderr)

    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")

    return completed

## 🧪 Step 3 — Create a virtual environment

A virtual environment keeps your project neat.
It means your workshop tools stay together instead of getting mixed into your whole computer.


In [8]:
# import venv
#
# if venv_python.exists():
#     print('✅ Virtual environment already exists')
# else:
#     print('Creating virtual environment...')
#     venv.create(venv_dir, with_pip=True)
#     print(f'✅ Created: {venv_dir}')

if venv_python.exists():
    print("✅ Virtual environment already exists")
    
else:
    print("Creating virtual environment...")
    result = run_command(
    	[sys.executable, "-m", "venv", venv_dir], 
    	cwd=repo_root
    )
    
    if result.returncode == 0:
        print("✅ Created: {venv_dir}")
    else:
        print("❌ Could not create the virtual environment")      
        
print(f'Python inside environment: {venv_python}')

✅ Virtual environment already exists
Python inside environment: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/.venv/bin/python


### ✅ Checkpoint
You should see either:
- `Virtual environment already exists`
or
- `Created: .../.venv`


## ▶️ Step 4 — Activate the environment

👉 This step is done in the **terminal**, not in a notebook cell.

**You do not have to activate it to continue this notebook**, but it is a good habit to learn. 

The following cell will print the appropriate command, to be copied and pasted, for your operating system.


In [9]:
print("👉 Activate the environment in your terminal using one of these commands:\n")

if OS_NAME == "Windows":
    print(r".\.venv\Scripts\activate")
    print("\nYou can still continue in this notebook even if you do not activate it now,")
    print("because the notebook uses the environment paths directly.")

else:
    print("source .venv/bin/activate")
    print("\nYou can still continue in this notebook even if you do not activate it now,")
    print("because the notebook uses the environment paths directly.")
    print()
    print("note:    if your terminal is engaged with the 'Jupyter Notebook' process,")
    print("\t you will not be able to run the command, in that case ignore.")

👉 Activate the environment in your terminal using one of these commands:

source .venv/bin/activate

You can still continue in this notebook even if you do not activate it now,
because the notebook uses the environment paths directly.

note:    if your terminal is engaged with the 'Jupyter Notebook' process,
	 you will not be able to run the command, in that case ignore.


### ✅ Checkpoint

Your terminal should now show something like:

`(venv)`

> You can still continue with this notebook even if you do not activate it right now, because the notebook uses the environment paths directly.
> But it is still good practice to understand this step.

---

## 📦 Step 5 — Install the workshop packages

This installs everything needed for the workshop. **Note**: it may take a few minutes to complete!

> We use `requirements_locked.txt` when available because it gives a more stable conference setup.
> That means fewer surprise version problems.


In [30]:
print("Installing project packages... this may take few minutes.")

run_command(
  [venv_python, "-m", "pip", "install", "--upgrade", "pip"],
  cwd=repo_root
)

install_result = run_command(
  [venv_pip, "install", "-r", requirements_file],
  cwd=repo_root
)

if install_result.returncode == 0:
    print("✅ Packages installed")
    print()
    print(requirements_file.read_text(encoding="utf-8"))

else:
    print("❌ Package installation failed")
    print("If you are on Python 3.12, check that requirements_locked.txt does not pin old packages such as pandas==1.5.3.")


Installing project packages... this may take few minutes.
Running: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/.venv/bin/python -m pip install --upgrade pip

Running: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/.venv/bin/pip install -r /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/requirements_locked.txt
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/13.0 MB ? eta -:--:--
   ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/13.0 MB 14.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━ 5.8/13.0 MB 19.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━ 10.5/13.0 MB 19.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 12.8/13.0 MB 20.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 15.8 MB/s  0:00:00
Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [tzdata]
  Attempting uninstall: pandas
 

### ✅ Checkpoint

🚦 You may have to wait several minutes for the previous cell to complete!

In the output window, if you scroll down to the bottom, you should see:

- `✅ Packages installed`
- package names including **dbt-duckdb** and **duckdb** somewhere in the install output


❌ If this fails:
- check your internet connection
- run the cell again
- make sure `requirements.txt` exists in the repo root


## 🔎 Step 6 — Verify the main tools

Let’s check that the environment can run Python, pip, and DBT.


In [11]:
print("Checking Python, pip, and dbt...\n")

run_command([venv_python, "--version"])
run_command([venv_pip, "--version"])
run_command([venv_dbt, "--version"])

print("✅ Core tools are available")


Checking Python, pip, and dbt...

Running: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/.venv/bin/python --version
Python 3.11.2

Running: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/.venv/bin/pip --version
pip 26.1 from /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/.venv/lib/python3.11/site-packages/pip (python 3.11)

Running: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/.venv/bin/dbt --version
Core:
  - installed: 1.11.8
  - latest:    1.11.8 - Up to date!

Plugins:
  - duckdb: 1.10.1 - Up to date!



✅ Core tools are available


### ✅ Checkpoint

You should see version information for:

- Python
- pip
- dbt

> Seeing a message about a newer DBT version is **not** a problem here.
> For a workshop, consistency is more important than always being newest.

---



## 🏗️ Step 7 — Build your DBT project structure

Normally, many people start with `dbt init`, but that command is interactive.

For this workshop, we will build the project step by step so you can clearly see what each piece does.


In [12]:
from pathlib import Path

folders_to_create = [
    "models",
    "seeds",
    "tests",
    "analysis",
    "macros"
]

for folder in folders_to_create:
    (dbt_project_dir / folder).mkdir(parents=True, exist_ok=True)

print("✅ DBT project folders created at:", dbt_project_dir)


✅ DBT project folders created at: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup



### ✅ Checkpoint

You should now have a folder structure like:

```text
├── dbt_projects/
│   └── p00_setup/
│       ├── models/
│       ├── seeds/
│       ├── tests/
│       ├── analysis/
│       └── macros/
```

That means your DBT project structure now exists locally.

---

In [13]:
print("Here is your DBT project structure so far:\n")

for item in sorted(dbt_project_dir.iterdir()):
    print("-", item.name)

Here is your DBT project structure so far:

- .user.yml
- analysis
- dbt_project.yml
- logs
- macros
- models
- profiles.yml
- seeds
- target
- tests
- workshop.duckdb



## 📄 Step 8 — Create `dbt_project.yml`

This is the main DBT project configuration file.

It tells DBT:

- what the project is called
- where your models live
- where seeds live
- how models should be built


In [26]:

import textwrap

dbt_project_yml = dbt_project_dir / "dbt_project.yml"

if dbt_project_yml.exists():
    print("✅ dbt_project.yml file already exists")
    
else:
    dbt_project_yml.write_text(
        textwrap.dedent(f"""\
        name: '{project_name}'
        version: '1.0'
        config-version: 2

        profile: '{project_name}'

        model-paths: ["models"]
        analysis-paths: ["analysis"]
        test-paths: ["tests"]
        seed-paths: ["seeds"]
        macro-paths: ["macros"]
        snapshot-paths: ["snapshots"]

        clean-targets:
          - "target"
          - "dbt_packages"

        models:
          {project_name}:
              +materialized: view

        """),
        encoding="utf-8"
    )

    print("✅ Created:", dbt_project_yml)
    
print()
print(dbt_project_yml.read_text(encoding="utf-8"))


✅ Created: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup/dbt_project.yml

name: 'p00_setup'
version: '1.0'
config-version: 2

profile: 'p00_setup'

model-paths: ["models"]
analysis-paths: ["analysis"]
test-paths: ["tests"]
seed-paths: ["seeds"]
macro-paths: ["macros"]
snapshot-paths: ["snapshots"]

clean-targets:
  - "target"
  - "dbt_packages"

models:
  p00_setup:
      +materialized: view





### ✅ Checkpoint

You should now have a file:

`dbt_projects/p00_setup/dbt_project.yml`

This file is the **heart** of your DBT project.



## 🔌 Step 9 — Create `profiles.yml`

DBT also needs a separate file that tells it **how to connect to a database**.

For this workshop, we use **DuckDB** because it is small, local, and easy to set up.


In [15]:


profiles_yml = dbt_project_dir / "profiles.yml"
database_path = dbt_project_dir / "workshop.duckdb"

if profiles_yml.exists():
    print("✅ Profile.yml file already exists")
    
else:
    profiles_yml.write_text(
        f"""{project_name}:
      target: dev
      outputs:
        dev:
          type: duckdb
          path: "{database_path.as_posix()}"
          threads: 1
    """,
        encoding="utf-8"
    )

    print('✅ Profile created')
    
print()
print(f'Project name: {project_name}')
print(f'Profile file: {profiles_yml}')
print(f'Database file: {database_path}')

print()
print(profiles_yml.read_text(encoding="utf-8"))


✅ Profile.yml file already exists

Project name: p00_setup
Profile file: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup/profiles.yml
Database file: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup/workshop.duckdb

p00_setup:
      target: dev
      outputs:
        dev:
          type: duckdb
          path: "/home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup/workshop.duckdb"
          threads: 1
    


### ✅ Checkpoint

You should see:
- `Profile created`
- your DBT project name
- a DuckDB file path name ending in `workshop.duckdb` written in the `profiles.yml` file

Also, you should now have:

`dbt_projects/p00_setup/profiles.yml`

This file connects your DBT project to a local DuckDB database.



## 📊 Step 10 — Create a small Brighton-themed dataset

A **seed** is just a CSV file that DBT can load into the database.

This one is a simple example dataset based on spending activity in different Brighton areas.


In [23]:

seed_file = dbt_project_dir / "seeds" / "example_brighton_spending.csv"

if seed_file.exists():
    print("✅ example_brighton_spending.csv seed file already exists")
    
else:
    seed_file.write_text(
    textwrap.dedent("""\
    transaction_id,area,venue_type,age_group,amount_spent
    1,Brighton Centre,Cafe,16-18,8.50
    2,North Laine,Bookshop,16-18,12.00
    3,Hove,Supermarket,19-21,24.50
    4,Kemptown,Cafe,16-18,7.20
    5,Brighton Marina,Entertainment,19-21,18.00
    6,North Laine,Cafe,16-18,6.80
    7,Hove,Restaurant,19-21,21.00
    8,Kemptown,Convenience Store,16-18,5.40
    9,Brighton Centre,Restaurant,19-21,19.50
    10,Brighton Marina,Cafe,16-18,9.10
    11,North Laine,Entertainment,19-21,14.00
    12,Hove,Cafe,16-18,6.50
    13,Kemptown,Restaurant,19-21,17.80
    14,Brighton Centre,Bookshop,16-18,11.20
    15,Brighton Marina,Restaurant,19-21,22.40
    16,North Laine,Supermarket,19-21,26.00
    17,Hove,Bookshop,16-18,10.50
    18,Kemptown,Entertainment,19-21,16.30
    19,Brighton Centre,Convenience Store,16-18,4.90
    20,Brighton Marina,Supermarket,19-21,28.00
    """).lstrip(),
    encoding="utf-8"
    )
  
    print("✅ Created seed file")
    
print()
print(seed_file.read_text(encoding="utf-8"))


✅ Created seed file

transaction_id,area,venue_type,age_group,amount_spent
1,Brighton Centre,Cafe,16-18,8.50
2,North Laine,Bookshop,16-18,12.00
3,Hove,Supermarket,19-21,24.50
4,Kemptown,Cafe,16-18,7.20
5,Brighton Marina,Entertainment,19-21,18.00
6,North Laine,Cafe,16-18,6.80
7,Hove,Restaurant,19-21,21.00
8,Kemptown,Convenience Store,16-18,5.40
9,Brighton Centre,Restaurant,19-21,19.50
10,Brighton Marina,Cafe,16-18,9.10
11,North Laine,Entertainment,19-21,14.00
12,Hove,Cafe,16-18,6.50
13,Kemptown,Restaurant,19-21,17.80
14,Brighton Centre,Bookshop,16-18,11.20
15,Brighton Marina,Restaurant,19-21,22.40
16,North Laine,Supermarket,19-21,26.00
17,Hove,Bookshop,16-18,10.50
18,Kemptown,Entertainment,19-21,16.30
19,Brighton Centre,Convenience Store,16-18,4.90
20,Brighton Marina,Supermarket,19-21,28.00



### ✅ Checkpoint

You should now have:

`dbt_projects/p00_setup/seeds/example_brighton_spending.csv`

Nice — you now have raw data ready to load.



## 🧮 Step 11 — Create your first DBT model

A **model** is an SQL file that transforms data.

This one summarises activity by Brighton area so we can compare places more easily.


In [17]:

models_dir = dbt_project_dir / "models" 
models_dir.mkdir(parents=True, exist_ok=True)

model_sql = models_dir / "example_brighton_area_summary.sql"

if model_sql.exists():
    print("✅ example_brighton_area_summary.sql model file already exists")
    
else:
    model_sql.write_text(
    """select
      area,
      sum(amount_spent) as total_spent,
      count(*) as total_transactions,
      round(avg(amount_spent), 2) as average_transaction
    from {{ ref('example_brighton_spending') }}
    group by area
    order by total_spent desc
    """,
    encoding="utf-8"
    )

    print("✅ Created:", model_sql)
    
print()
print(model_sql.read_text(encoding="utf-8"))


✅ example_brighton_area_summary.sql model file already exists

select
      area,
      sum(amount_spent) as total_spent,
      count(*) as total_transactions,
      round(avg(amount_spent), 2) as average_transaction
    from {{ ref('example_brighton_spending') }}
    group by area
    order by total_spent desc
    


### ✅ Checkpoint

You should now have:

`example_brighton_area_summary.sql`

This is your first transformation model.

## 🛡️ Step 12 — Add Data Quality Checks
  
Before we trust data, we should test it.  
  
That’s one of the really useful things DBT does well ✅  
  
In this step, you will add rules to check that:  
  
- 💷 `amount_spent` is greater than zero
- 👥 `age_group` is one of the allowed values
  
Think of this as a **quality gate** for your data.  
  
If the data breaks the rules, DBT will tell you.

---


## 🧩 Step 12a — Create a Custom Test Rule
  
  DBT already includes some built-in tests like:  
  
- `not_null`
- `unique`
- `accepted_values`
  
  But for:  
  
  >   “amount spent must be greater than zero”  
  
  we will create our own small reusable test.

---

In [18]:
macro_file = dbt_project_dir / "macros" / "test_positive_value.sql"

if macro_file.exists():
    print("✅ Generic test positive value macro file already exists")
    
else:
    macro_file.write_text(
    """{% test positive_value(model, column_name) %}

    select *
    from {{ model }}
    where {{ column_name }} <= 0

    {% endtest %}
    """,
    encoding="utf-8"
    )

    print("✅ Generic test positive value macro created")
    
print()
print(macro_file.read_text(encoding="utf-8"))



✅ Generic test positive value macro file already exists

{% test positive_value(model, column_name) %}

    select *
    from {{ model }}
    where {{ column_name }} <= 0

    {% endtest %}
    


### ✅ Checkpoint
  
  You should see: 
  
- a success message
- a path ending in:
  
  ```
  dbt_projects/p00_setup/macros/test_positive_value.sql
  ```
- ### 💡 What just happened?
  
  You created a reusable DBT test called:  
  
  ```
  positive_value
  ```
  
  It checks for rows where a numeric value is **zero or less**.  
  
  In our case, that means:  
  
  >   “show me any rows where `amount_spent` is not valid”

---

## 📝 Step 12b — Create  `setup_schema.yml` for applying Validation Rules 
  
  Now we define the actual rules DBT should apply to the data.  
  
  This file tells DBT:  
  
- which table to test
- which columns to check
- which rules must pass

---

In [19]:
import textwrap

schema_file = models_dir / "setup_schema.yml"

if schema_file.exists():
    print("✅ setup_schema.yml Validation Rules file already exists")
    
else:
    schema_file.write_text(
        textwrap.dedent("""\
        version: 2

        seeds:
          - name: example_brighton_spending
            description: "Small Brighton-themed spending dataset used for workshop practice."
            columns:
              - name: transaction_id
                description: "Unique id for each transaction"
                tests:
                  - not_null
                  - unique

              - name: amount_spent
                description: "Amount spent in pounds"
                tests:
                  - not_null
                  - positive_value

              - name: age_group
                description: "Age band of the customer"
                tests:
                  - not_null
                  - accepted_values:
                      values: ['16-18', '19-21', '22-25', '26-30', '31+']

        models:
          - name: example_brighton_area_summary
            description: "Summary of spending activity by Brighton area."
            columns:
              - name: area
                description: "Area name"
                tests:
                  - not_null

              - name: total_spent
                description: "Total spending in each area"
                tests:
                  - not_null
        """),
        encoding="utf-8"
    )    
    print("✅ setup_schema.yml created")
    
print()
print(schema_file.read_text(encoding="utf-8"))


✅ setup_schema.yml Validation Rules file already exists

version: 2

seeds:
  - name: example_brighton_spending
    description: "Small Brighton-themed spending dataset used for workshop practice."
    columns:
      - name: transaction_id
        description: "Unique id for each transaction"
        tests:
          - not_null
          - unique

      - name: amount_spent
        description: "Amount spent in pounds"
        tests:
          - not_null
          - positive_value

      - name: age_group
        description: "Age band of the customer"
        tests:
          - not_null
          - accepted_values:
              values: ['16-18', '19-21', '22-25', '26-30', '31+']

models:
  - name: example_brighton_area_summary
    description: "Summary of spending activity by Brighton area."
    columns:
      - name: area
        description: "Area name"
        tests:
          - not_null

      - name: total_spent
        description: "Total spending in each area"
        tests:
 

### ✅ Checkpoint
  
  You should see:  
  
- a success message
- the contents of `schema.yml` printed in the notebook

### 🔍 Rules you just added
  
  For the seed table:  
- `transaction_id` must not be empty
- `transaction_id` must be unique
- `amount_spent` must not be empty
- `amount_spent` must be greater than zero
- `age_group` must be one of:
	- `16-18`
	- `19-21`
	- `22-25`
	- `26-30`
	- `31+`
	    
	  For the final model:  
- `area` must not be empty
- `total_spent` must not be empty

### 💡 Why this matters
  
  This is where DBT becomes more than just SQL.  
  
  It helps you say:  
  
  >   “These are the rules my data must follow.”  
  
  That is real-world analytics thinking.

---

## 🧪 Step 13 — Test the DBT setup with `dbt debug`

Now we check whether:
- the profile is valid
- the project is valid
- DBT can connect to DuckDB


In [20]:
debug_result = run_command(
  [
    venv_dbt,
    'debug',
    '--project-dir', dbt_project_dir,
    '--profiles-dir', dbt_project_dir
  ],
  cwd=repo_root, check=False
)

if debug_result.returncode == 0:
    print('✅ DBT debug passed')
else:
    raise RuntimeError('❌ DBT debug failed. Scroll up and read the output carefully.')


Running: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/.venv/bin/dbt debug --project-dir /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup --profiles-dir /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup
17:50:39  Running with dbt=1.11.8
17:50:39  dbt version: 1.11.8
17:50:39  python version: 3.11.2
17:50:39  python path: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/.venv/bin/python
17:50:39  os info: Linux-6.6.99-09128-g14e87a8a9b71-x86_64-with-glibc2.36
17:50:39  Using profiles dir at /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup
17:50:39  Using profiles.yml file at /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup/profiles.yml
17:50:39  Using dbt_project.yml file at /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup/dbt_project.yml
17:50:39  adapter type: duckdb
17:50:39  adapter version: 1.10.1
17:50:39  Configuration:
17:50:39    profiles.yml file [

### ✅ Checkpoint
You want to see a successful DBT debug result.

The most important signs are:

- the profile is found
- the project is found
- the connection is OK

❌ If this fails:
- check that `dbt_projects/p00_setup/dbt_project.yml` exists
- check that `dbt_projects/p00_setup/profiles.yml` exists
- re-run Step 6 and Step 7


## 🌱 Step 14 — Load the seed data into DuckDB

A **seed** is just a CSV file that DBT loads into the database for you.

In this workshop, that means your raw data gets pulled into DuckDB, ready for modelling.


In [24]:
seed_result = run_command(
  [
    venv_dbt,
    'seed',
    '--project-dir', dbt_project_dir,
    '--profiles-dir', dbt_project_dir
  ],
  cwd=repo_root, check=False
)

if seed_result.returncode == 0:
    print('✅ Seed loaded successfully')
else:
    raise RuntimeError('❌ DBT seed loading failed. Scroll up and read the output carefully.')


Running: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/.venv/bin/dbt seed --project-dir /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup --profiles-dir /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup
17:57:26  Running with dbt=1.11.8
17:57:26  Registered adapter: duckdb=1.10.1
17:57:27  [WARNING][MissingArgumentsPropertyInGenericTestDeprecation]: Deprecated
functionality
Found top-level arguments to test `accepted_values` defined on
'example_brighton_spending' in package 'p00_setup' (models/setup_schema.yml).
Arguments to generic tests should be nested under the `arguments` property.
17:57:27  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 2 unused configuration paths:
- models.p00_setup.marts
- models.p00_setup.staging
17:57:27  Found 1 model, 8 data tests, 1 seed, 476 macros
17:57:27  
17:57:27  Concurrency: 1 threads (target='dev')
17:57:27  
17:57:28  1 of 1 STA

### ✅ Checkpoint
You should see a success message for the seed step.

That means the CSV file has been loaded into your **DuckDB** database.


## 🧱 Step 15 — Run the transformation model

This is the big moment.

DBT now takes the raw data and builds cleaner, more useful tables from it.


In [27]:
run_result = run_command(
  [
    venv_dbt,
    'run',
    '--select', 'example_brighton_area_summary',   
    '--project-dir', dbt_project_dir,
    '--profiles-dir', dbt_project_dir
  ],
  cwd=repo_root, check=False
)

if run_result.returncode == 0:
    print('✅ DBT run completed successfully')
else:
    raise RuntimeError('❌ DBT run failed. Scroll up and read the output carefully.')


Running: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/.venv/bin/dbt run --select example_brighton_area_summary --project-dir /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup --profiles-dir /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup
18:18:05  Running with dbt=1.11.8
18:18:06  Registered adapter: duckdb=1.10.1
18:18:06  Unable to do partial parsing because a project config has changed
18:18:08  [WARNING][MissingArgumentsPropertyInGenericTestDeprecation]: Deprecated
functionality
Found top-level arguments to test `accepted_values` defined on
'example_brighton_spending' in package 'p00_setup' (models/setup_schema.yml).
Arguments to generic tests should be nested under the `arguments` property.
18:18:09  Found 1 model, 1 seed, 8 data tests, 476 macros
18:18:09  
18:18:09  Concurrency: 1 threads (target='dev')
18:18:09  
18:18:09  1 of 1 START sql view model main.example_brighton_area_summary ................. [RUN]
18:18:09  1 of

### ✅ Checkpoint
You should see success output from DBT.

Depending on your project, you should now have a transformed model such as:
- `example_brighton_area_summary`


## 🧪 Step 16 — Check your Data against the Rules with 'dbt test'

Now let DBT check your data against the rules.

If everything is valid, the tests should pass ✅

If something is wrong, DBT will show you exactly which rule failed ❌

---


In [28]:

test_result = run_command(
  [
    venv_dbt,
    'test',
    '--select', 'setup',   
    '--project-dir', dbt_project_dir,
    '--profiles-dir', dbt_project_dir
  ],
  cwd=repo_root, check=False
)

if test_result.returncode == 0:
    print('✅ DBT tests passed')
else:
    print('⚠️ Some tests failed. That does not always mean your install is broken, but it is worth checking.')


Running: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/.venv/bin/dbt test --select setup --project-dir /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup --profiles-dir /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup
18:22:42  Running with dbt=1.11.8
18:22:42  Registered adapter: duckdb=1.10.1
18:22:43  Found 1 model, 1 seed, 8 data tests, 476 macros
18:22:43  The selection criterion 'setup' does not match any enabled nodes
18:22:43  Nothing to do. Try checking your model configs and model specification args

✅ DBT tests passed


### ✅ Checkpoint
  
  You want to see: 
  
- DBT test output
- tests running successfully
- a final message 'DBT test passed'
  
### 🎉 If the tests pass
  
  That means your data meets the rules you defined.  

### ❗ If the tests fail
  
  That means DBT found a data quality issue.  
  
  That is not a disaster — it is actually useful.  
  
  It means the tests are doing their job.

---


## 🔍 Step 17 — Query your result

Now let’s look at the table you created.

> This cell uses the Python inside your virtual environment, so it works even if your notebook kernel is different.


In [31]:
query_code = textwrap.dedent(f'''
from pathlib import Path
import duckdb
import pandas as pd  # 👈 not strictly required, but clearer for learners

db_path = Path(r"{database_path}")
con = duckdb.connect(str(db_path))

query = con.execute("SELECT * FROM example_brighton_area_summary").fetchdf()

print(query.to_string(index=False))
''')

# query_result = run_command([venv_python, '-c', query_code], cwd=repo_root)

query_result = run_command([
    venv_python, 
    '-c', query_code
], cwd=repo_root, check=False)


# if query_result.returncode != 0:
#    print("❌ Query failed")
    
if query_result.returncode == 0:
    print('✅ Query result successful')
else:
    print('❌ Query failed. Scroll up and read the output carefully.')
 


Running: /home/michaelliedl/Prj/Data-Analytics/dbt-workshop/.venv/bin/python -c 
from pathlib import Path
import duckdb
import pandas as pd  # 👈 not strictly required, but clearer for learners

db_path = Path(r"/home/michaelliedl/Prj/Data-Analytics/dbt-workshop/dbt_projects/p00_setup/workshop.duckdb")
con = duckdb.connect(str(db_path))

query = con.execute("SELECT * FROM example_brighton_area_summary").fetchdf()

print(query.to_string(index=False))

           area  total_spent  total_transactions  average_transaction
Brighton Marina         77.5                   4                19.38
           Hove         62.5                   4                15.63
    North Laine         58.8                   4                14.70
       Kemptown         46.7                   4                11.68
Brighton Centre         44.1                   4                11.03

✅ Query result successful


### ✅ Checkpoint
You should see a small table of results.

Nice — you are now reading from a database table that **you created**.


## 🧠 What You Learned Here

You just used DBT to check **data quality**, not just transform data.

That matters because in real projects, analysts and engineers need to trust the data before making decisions from it.

So the workflow is not just:

**load → transform**

It is:

**load → validate → transform → trust**



## 🛠️ Troubleshooting

### Problem: package install failed
- check your internet connection
- run the install step again

### Problem: `dbt` is not found
Run Step 2 again. That usually means the package install did not finish properly.

### Problem: `DBT debug failed`
Check these carefully:
- are you inside the correct repo?
- does `dbt_projects/p00_setup/dbt_project.yml` exist?
- was `profiles.yml` created in your home `dbt_projects/p00_setup` folder?

re-run the steps that create:
- `dbt_projects/p00_setup/dbt_project.yml`
- `dbt_projects/p00_setup/profiles.yml

### Problem: nothing seems to happen
- check that you are inside the correct workshop folder
- restart Jupyter and reopen this notebook

### Problem: permission errors on Linux or Chromebook
Close the notebook, open a fresh terminal, and run Jupyter again from your own user account.

### Problem: you want to rebuild from scratch
Run the next cell carefully. It removes the project environment and the DBT project folder.  

Then go back to Step 1.

> Only use the reset cell if you really want a clean restart.



In [ ]:

import shutil

for path in [repo_root / "venv", repo_root / "dbt_projects"]:
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
            print("Removed folder:", path)
        else:
            path.unlink()
            print("Removed file:", path)
    else:
        print("Not found:", path)



## 🧠 What you achieved

You just:

- created a Python environment ✅
- installed DBT and DuckDB ✅
- built a DBT project structure ✅
- created `dbt_project.yml` ✅
- created `profiles.yml` ✅
- loaded seed data into DuckDB✅
- built a transformed table ✅
- queried your results ✅

That is a real end-to-end data workflow — not just theory.


## 🎉 You did it

You now have a working DBT setup and a working local data pipeline.

## ✅ Next step
Open the project notebook in:

`03_project/From_Data_to_Decision.ipynb`

That is where you will:
- explore patterns 📊
- build insights 💡
- turn data into decisions 🚀
